In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("data/used_cars.csv")

# Display first 5 rows
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [2]:
# Summary statistics
print("Numerical Features:")
display(df.describe())

print("\nCategorical Features:")
display(df.describe(include="object"))


Numerical Features:


,model_year
count,4009.000000
mean,2015.515590
std,6.104816
min,1974.000000
25%,2012.000000
50%,2017.000000
75%,2020.000000
max,2024.000000



Categorical Features:


,brand,model,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
count,4009,4009,4009,3839,4009,4009,4009,4009,3896,3413,4009
unique,57,1898,2818,7,1146,62,319,156,2,1,1569
top,Ford,M3 Base,"110,000 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,A/T,Black,Black,None reported,Yes,"$15,000"
freq,386,30,16,3309,52,1037,905,2025,2910,3413,39


In [3]:
# Handle missing values

# Fill missing numerical values with median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill missing categorical values with mode
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining Missing Values:")
print(df.isnull().sum())

Remaining Missing Values:
brand           0
model           0
model_year      0
milage          0
fuel_type       0
engine          0
transmission    0
ext_col         0
int_col         0
accident        0
clean_title     0
price           0
dtype: int64


In [4]:
# Remove duplicate rows
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates()

print("Duplicates removed:", duplicates_before)
print("New dataset shape:", df.shape)

Duplicates removed: 0
New dataset shape: (4009, 12)


In [5]:
# Feature Engineering

# Car Age (assuming current year is 2026)
if 'year' in df.columns:
    df['car_age'] = 2026 - df['year']

# Price per Kilometer
if 'price' in df.columns and 'kms_driven' in df.columns:
    df['price_per_km'] = df['price'] / (df['kms_driven'] + 1)

# Mileage Category
if 'mileage' in df.columns:
    df['mileage_category'] = pd.cut(
        df['mileage'],
        bins=[0, 15, 20, 100],
        labels=['Low', 'Medium', 'High']
    )

# Engine Category
if 'engine' in df.columns:
    df['engine_category'] = pd.cut(
        df['engine'],
        bins=[0, 1200, 1800, 10000],
        labels=['Small', 'Medium', 'Large']
    )

# Power-to-Engine Ratio
if 'max_power' in df.columns and 'engine' in df.columns:
    df['power_engine_ratio'] = df['max_power'] / (df['engine'] + 1)

print(df.head())

TypeError: '<' not supported between instances of 'int' and 'str'

In [6]:
df.dtypes

brand           object
model           object
model_year       int64
milage          object
fuel_type       object
engine          object
transmission    object
ext_col         object
int_col         object
accident        object
clean_title     object
price           object
dtype: object

In [7]:
print(df.dtypes)

brand           object
model           object
model_year       int64
milage          object
fuel_type       object
engine          object
transmission    object
ext_col         object
int_col         object
accident        object
clean_title     object
price           object
dtype: object


In [8]:

print(df.dtypes)

brand           object
model           object
model_year       int64
milage          object
fuel_type       object
engine          object
transmission    object
ext_col         object
int_col         object
accident        object
clean_title     object
price           object
dtype: object


In [9]:
print(df.dtypes)

brand           object
model           object
model_year       int64
milage          object
fuel_type       object
engine          object
transmission    object
ext_col         object
int_col         object
accident        object
clean_title     object
price           object
dtype: object


In [10]:
print(df.dtypes)

brand           object
model           object
model_year       int64
milage          object
fuel_type       object
engine          object
transmission    object
ext_col         object
int_col         object
accident        object
clean_title     object
price           object
dtype: object


In [12]:
# Feature Engineering

# Convert price to numeric
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Car Age
df["car_age"] = 2026 - df["model_year"]

# Premium Car (Above Median Price)
df["premium_car"] = df["price"] > df["price"].median()

# Brand Frequency
df["brand_count"] = df.groupby("brand")["brand"].transform("count")

# Luxury Brand Flag
luxury_brands = ["BMW", "Mercedes-Benz", "Audi", "Lexus", "Porsche", "Land", "Jaguar"]
df["is_luxury"] = df["brand"].str.contains("|".join(luxury_brands), case=False, na=False)

# Price Category
df["price_category"] = pd.qcut(
    df["price"],
    q=3,
    labels=["Budget", "Mid-Range", "Premium"]
)

print(df.head())

      brand                            model  model_year      milage  \
0      Ford  Utility Police Interceptor Base        2013  51,000 mi.   
1   Hyundai                     Palisade SEL        2021  34,742 mi.   
2     Lexus                    RX 350 RX 350        2022  22,372 mi.   
3  INFINITI                 Q50 Hybrid Sport        2015  88,900 mi.   
4      Audi        Q3 45 S line Premium Plus        2021   9,835 mi.   

       fuel_type                                             engine  \
0  E85 Flex Fuel  300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...   
1       Gasoline                               3.8L V6 24V GDI DOHC   
2       Gasoline                                     3.5 Liter DOHC   
3         Hybrid  354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...   
4       Gasoline                         2.0L I4 16V GDI DOHC Turbo   

        transmission                 ext_col int_col  \
0        6-Speed A/T                   Black   Black   
1  8-Speed Automatic        

In [13]:
# Save cleaned dataset
df.to_csv("cleaned_used_cars.csv", index=False)

print("✅ Cleaned dataset saved successfully!")
print("Final Dataset Shape:", df.shape)

✅ Cleaned dataset saved successfully!
Final Dataset Shape: (4009, 17)


In [14]:
# Five Business Insights

print("1. Top 10 Brands:")
print(df["brand"].value_counts().head(10))

print("\n2. Fuel Type Distribution:")
print(df["fuel_type"].value_counts())

print("\n3. Average Price by Brand:")
print(df.groupby("brand")["price"].mean().sort_values(ascending=False).head(10))

print("\n4. Average Price by Transmission:")
print(df.groupby("transmission")["price"].mean())

print("\n5. Average Price by Car Age:")
print(df.groupby("car_age")["price"].mean().sort_index().head(10))

1. Top 10 Brands:
brand
Ford             386
BMW              375
Mercedes-Benz    315
Chevrolet        292
Porsche          201
Audi             200
Toyota           199
Lexus            163
Jeep             143
Land             130
Name: count, dtype: int64

2. Fuel Type Distribution:
fuel_type
Gasoline          3479
Hybrid             194
E85 Flex Fuel      139
Diesel             116
–                   45
Plug-In Hybrid      34
not supported        2
Name: count, dtype: int64

3. Average Price by Brand:
brand
Bugatti        1.950995e+06
Rolls-Royce    3.709927e+05
Lamborghini    2.912338e+05
Ferrari        2.437907e+05
McLaren        2.134575e+05
Maserati       1.405825e+05
Bentley        1.375535e+05
Aston          1.151996e+05
Lucid          1.019663e+05
Rivian         9.313818e+04
Name: price, dtype: float64

4. Average Price by Transmission:
transmission
1-Speed A/T                           51731.750000
1-Speed Automatic                     51076.571429
10-Speed A/T           

## 8. Business Insights and Summary

### Key Insights

1. **Popular Brands:** BMW, Mercedes-Benz, Chevrolet, Porsche, and Audi are among the most frequently listed brands, indicating strong demand in the used car market.

2. **Fuel Type Distribution:** Gasoline vehicles dominate the used car market, while Hybrid, Diesel, and E85 Flex Fuel vehicles account for a smaller share.

3. **Premium Brands:** Luxury brands generally have higher average selling prices than mass-market brands, reflecting stronger resale value.

4. **Transmission Impact:** Vehicles with automatic transmissions generally command higher average selling prices than manual transmission vehicles.

5. **Vehicle Age vs. Price:** Newer vehicles typically have higher resale values, while older vehicles depreciate over time.